# Notebook 03 -- Statistical Analysis: Retrieval Metrics, BERTScore, and Significance Tests

This notebook performs the full statistical evaluation of the 6-config ablation
study. It runs independently of Notebook 02 -- it only reads the prediction
files that Notebook 02 saved to Drive.

**Sections:**
1. Retrieval-only evaluation (Recall@K, MRR, nDCG@10) at multiple relevance thresholds
2. BERTScore computation for all 6 configs
3. Bootstrap 95% confidence intervals for Token F1, ROUGE-L, and BERTScore
4. Wilcoxon signed-rank tests for 6 key pairwise comparisons
5. Save all results to Drive

**Requires:** Prediction files from Notebook 02 on Drive at `results/config*_predictions*.json`.

In [ ]:
# ====================================================================
# 1. SETUP
# ====================================================================

# ---- Drive mount ----
from google.colab import drive, userdata
drive.mount("/content/drive")

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""

# ---- Installs ----
!pip install -q sentence-transformers faiss-cpu rank_bm25 bert-score rouge-score scipy tqdm pyarrow 2>/dev/null

# ---- Imports ----
import gc
import json
import os
import pickle
import re
import sys
import random
from collections import Counter
from pathlib import Path

import numpy as np
from scipy.stats import wilcoxon
from tqdm.auto import tqdm

# ---- Seeds ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- Paths ----
DRIVE_ROOT   = Path("/content/drive/MyDrive/hukuk-rag")
BM25_PATH    = DRIVE_ROOT / "indexes" / "bm25.pkl"
BASE_FAISS   = DRIVE_ROOT / "indexes" / "faiss.index"
BASE_MAP     = DRIVE_ROOT / "indexes" / "faiss.mapping.pkl"
FT_FAISS     = DRIVE_ROOT / "indexes" / "finetuned" / "faiss_ft.index"
FT_MAP       = DRIVE_ROOT / "indexes" / "finetuned" / "faiss_ft.mapping.pkl"
CHUNKS_PATH  = DRIVE_ROOT / "data" / "processed" / "chunks_filtered.parquet"
FT_E5_PATH   = DRIVE_ROOT / "models" / "e5-checkpoints" / "checkpoint-10000"
RESULTS_DIR  = DRIVE_ROOT / "results"

PROJECT_ROOT = Path("/content/hukuk-rag")
GOLD_PATH    = PROJECT_ROOT / "data" / "gold" / "gold_test_set.json"

# ---- Clone repo for gold set ----
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/berkay-aktas/hukuk-rag.git"
if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git pull

# ---- Constants (inlined for Colab portability) ----
TURKISH_LOWER_MAP = str.maketrans("\u0130I\u00d6\u00dc\u00c7\u015e\u011e",
                                  "i\u0131\u00f6\u00fc\u00e7\u015f\u011f")

TURKISH_STOPWORDS = frozenset({
    "bir", "bu", "da", "de", "ve", "ile", "i\u00e7in", "olan", "olarak", "gibi",
    "daha", "en", "\u00e7ok", "her", "kadar", "sonra", "\u00f6nce", "ise", "ya",
    "ne", "nas\u0131l", "neden", "nerede", "kim", "hangi", "o", "\u015fu", "ben",
    "sen", "biz", "siz", "onlar", "mi", "mu", "m\u00fc", "m\u0131", "dir", "d\u0131r",
    "dur", "d\u00fcr", "tir", "t\u0131r", "tur", "t\u00fcr", "ki", "ama", "ancak",
    "fakat", "lakin", "veya", "yahut", "hem", "\u00fczere", "g\u00f6re", "kar\u015f\u0131",
    "aras\u0131nda", "taraf\u0131ndan", "dolay\u0131", "halde", "ra\u011fmen", "itibaren",
    "de\u011fil", "var", "yok", "eden", "eder", "etti", "oldu",
    "olur", "olmu\u015f", "iken", "olup", "buna", "\u015f\u00f6yle",
    "b\u00f6yle", "\u00f6yle", "ayn\u0131", "baz\u0131", "bir\u00e7ok", "di\u011fer", "ba\u015fka",
})

_RE_PUNCT = re.compile(r'[^\w\s]')
_RE_MULTI_SPACE = re.compile(r'\s+')


def normalize_turkish(text):
    """Normalize Turkish text: locale-aware lowercase, remove punct, collapse whitespace."""
    text = text.translate(TURKISH_LOWER_MAP).lower()
    text = _RE_PUNCT.sub(' ', text)
    text = _RE_MULTI_SPACE.sub(' ', text).strip()
    return text


def turkish_tokenize(text):
    """Tokenize with Turkish lowercasing and stopword removal."""
    text = normalize_turkish(text)
    return [t for t in text.split() if t not in TURKISH_STOPWORDS and len(t) > 1]


def content_tokens(text):
    """Extract content-bearing token set (len>2, no stopwords) for relevance scoring."""
    norm = normalize_turkish(text)
    return {t for t in norm.split() if t not in TURKISH_STOPWORDS and len(t) > 2}


print("Setup complete.")

## Retrieval-Only Evaluation

Computes retrieval quality metrics for both the base E5 and fine-tuned E5
embedding models. Since the gold set does not have pre-labeled relevant
document IDs, we use a **proxy relevance** approach: a retrieved chunk is
considered relevant at a given threshold if the Jaccard overlap of its
content tokens with the gold answer's content tokens exceeds that threshold.

We evaluate at three thresholds (0.15, 0.25, 0.35) to show sensitivity.

Metrics: Recall@5, Recall@10, MRR, nDCG@10.

Additionally reports:
- **Madde-hit rate:** fraction of questions where a retrieved chunk contains
  the expected `madde_no` from the gold set.
- **Domain breakdown:** metrics split by question domain (criminal, civil, etc.).

In [ ]:
import faiss
import pyarrow.parquet as pq

# ---- Load gold data ----
with open(GOLD_PATH, "r", encoding="utf-8") as f:
    gold_raw = json.load(f)
gold_data = gold_raw["questions"]
print(f"Gold set: {len(gold_data)} questions")

# ---- Load BM25 ----
with open(BM25_PATH, "rb") as f:
    bm25_data = pickle.load(f)
bm25_idx = bm25_data["index"]
bm25_map = bm25_data["mapping"]
print(f"BM25: {len(bm25_map):,} chunks")

# ---- Load base FAISS index ----
base_faiss = faiss.read_index(str(BASE_FAISS))
with open(BASE_MAP, "rb") as f:
    base_ids = pickle.load(f)  # list of dicts: {'chunk_id': ..., 'text': ...}
print(f"Base FAISS: {base_faiss.ntotal:,} vectors")

# ---- Load FT FAISS index ----
ft_faiss = faiss.read_index(str(FT_FAISS))
with open(FT_MAP, "rb") as f:
    ft_ids = pickle.load(f)  # list of chunk_id strings
print(f"FT FAISS: {ft_faiss.ntotal:,} vectors")

# ---- Build chunk text list from parquet (for FT mapping text lookup) ----
print("Building chunk text list from parquet (streaming)...")
chunk_text_map = {}
pf = pq.ParquetFile(str(CHUNKS_PATH))
for batch in pf.iter_batches(batch_size=100_000, columns=["chunk_id", "text"]):
    df = batch.to_pandas()
    for cid, txt in zip(df["chunk_id"], df["text"]):
        chunk_text_map[cid] = txt
    del df
chunk_text_list = list(chunk_text_map.values())
print(f"Chunk text map: {len(chunk_text_map):,} entries")


# ---- Load base E5 model ----
from sentence_transformers import SentenceTransformer
print("Loading base E5...")
base_e5 = SentenceTransformer("intfloat/multilingual-e5-large")

# ---- Retrieval helpers ----

def run_dense_search(query, faiss_idx, mapping, model, k=50, nprobe=16):
    """Run FAISS dense search, returning list of (idx, score) pairs."""
    faiss_idx.nprobe = nprobe
    qvec = model.encode(
        [f"query: {query}"], normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = faiss_idx.search(qvec, k)
    return [(int(idx), float(s)) for s, idx in zip(scores[0], indices[0]) if idx != -1]


def get_text_for_result(idx, mapping, is_ft=False):
    """Get text for a FAISS result index. Base uses dicts, FT uses chunk_id strings."""
    if is_ft:
        cid = mapping[idx]
        return chunk_text_map.get(cid, "")
    else:
        return mapping[idx]["text"]


def proxy_relevance(retrieved_text, gold_answer, threshold):
    """Check if a retrieved chunk is relevant via content token Jaccard overlap."""
    r_tokens = content_tokens(retrieved_text)
    g_tokens = content_tokens(gold_answer)
    if not r_tokens or not g_tokens:
        return False
    overlap = len(r_tokens & g_tokens)
    jaccard = overlap / len(r_tokens | g_tokens)
    return jaccard >= threshold


def compute_retrieval_metrics(retrieved_texts, gold_answer, threshold, k_values=(5, 10)):
    """Compute Recall@K, MRR, nDCG@10 for a single query."""
    rels = [proxy_relevance(t, gold_answer, threshold) for t in retrieved_texts]

    results = {}
    for k in k_values:
        hits = sum(rels[:k])
        total_rel = max(sum(rels), 1)  # avoid division by zero
        results[f"recall@{k}"] = hits / total_rel

    # MRR
    mrr = 0.0
    for rank, rel in enumerate(rels, 1):
        if rel:
            mrr = 1.0 / rank
            break
    results["mrr"] = mrr

    # nDCG@10
    dcg = sum(1.0 / np.log2(rank + 2) for rank, rel in enumerate(rels[:10]) if rel)
    ideal_hits = min(sum(rels), 10)
    idcg = sum(1.0 / np.log2(rank + 2) for rank in range(ideal_hits))
    results["ndcg@10"] = dcg / idcg if idcg > 0 else 0.0

    return results


# ---- Run retrieval for both models ----
THRESHOLDS = [0.15, 0.25, 0.35]
K_DENSE = 50

# Storage: model -> threshold -> list of per-question metric dicts
retrieval_results = {"base": {t: [] for t in THRESHOLDS},
                     "ft": {t: [] for t in THRESHOLDS}}
madde_hits = {"base": 0, "ft": 0}
domain_metrics = {}  # domain -> model -> threshold -> list of metric dicts

print("\nRunning retrieval with base E5...")
for item in tqdm(gold_data, desc="Base E5"):
    q = item["question"]
    gold_ans = item["gold_answer"]
    domain = item.get("domain", "unknown")
    madde = item.get("madde_no", "")

    results = run_dense_search(q, base_faiss, base_ids, base_e5, k=K_DENSE)
    texts = [base_ids[idx]["text"] for idx, _ in results]

    # Madde hit check
    if madde:
        madde_pattern = f"Madde {madde}"
        if any(madde_pattern in t for t in texts[:10]):
            madde_hits["base"] += 1

    for thresh in THRESHOLDS:
        m = compute_retrieval_metrics(texts, gold_ans, thresh)
        retrieval_results["base"][thresh].append(m)

        if domain not in domain_metrics:
            domain_metrics[domain] = {
                "base": {t: [] for t in THRESHOLDS},
                "ft": {t: [] for t in THRESHOLDS}
            }
        domain_metrics[domain]["base"][thresh].append(m)

# Free base E5 before loading FT E5
del base_e5
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass
print("Base E5 freed.")

# ---- Load FT E5 ----
print("Loading FT E5...")
ft_e5 = SentenceTransformer(str(FT_E5_PATH))

print("Running retrieval with FT E5...")
for item in tqdm(gold_data, desc="FT E5"):
    q = item["question"]
    gold_ans = item["gold_answer"]
    domain = item.get("domain", "unknown")
    madde = item.get("madde_no", "")

    results = run_dense_search(q, ft_faiss, ft_ids, ft_e5, k=K_DENSE)
    texts = [chunk_text_map.get(ft_ids[idx], "") for idx, _ in results]

    # Madde hit check
    if madde:
        madde_pattern = f"Madde {madde}"
        if any(madde_pattern in t for t in texts[:10]):
            madde_hits["ft"] += 1

    for thresh in THRESHOLDS:
        m = compute_retrieval_metrics(texts, gold_ans, thresh)
        retrieval_results["ft"][thresh].append(m)

        if domain not in domain_metrics:
            domain_metrics[domain] = {
                "base": {t: [] for t in THRESHOLDS},
                "ft": {t: [] for t in THRESHOLDS}
            }
        domain_metrics[domain]["ft"][thresh].append(m)

# Free FT E5
del ft_e5
gc.collect()

# ---- Print multi-threshold table ----
n_with_madde = sum(1 for item in gold_data if item.get("madde_no", ""))

print("\n" + "=" * 80)
print("RETRIEVAL METRICS (Dense only, top-50 retrieved)")
print("=" * 80)

for thresh in THRESHOLDS:
    print(f"\n--- Threshold: {thresh} ---")
    header = f"{'Model':<12} {'Recall@5':>10} {'Recall@10':>10} {'MRR':>10} {'nDCG@10':>10}"
    print(header)
    print("-" * len(header))

    for model_name in ["base", "ft"]:
        metrics_list = retrieval_results[model_name][thresh]
        avg = {
            k: np.mean([m[k] for m in metrics_list])
            for k in ["recall@5", "recall@10", "mrr", "ndcg@10"]
        }
        label = "Base E5" if model_name == "base" else "FT E5"
        print(f"{label:<12} {avg['recall@5']:>10.4f} {avg['recall@10']:>10.4f} "
              f"{avg['mrr']:>10.4f} {avg['ndcg@10']:>10.4f}")

# ---- Madde-hit table ----
print(f"\n--- Madde Hit Rate (top-10 retrieved, {n_with_madde} questions with madde_no) ---")
print(f"{'Model':<12} {'Hits':>6} {'Rate':>10}")
print("-" * 30)
for model_name, label in [("base", "Base E5"), ("ft", "FT E5")]:
    hits = madde_hits[model_name]
    rate = hits / n_with_madde if n_with_madde > 0 else 0
    print(f"{label:<12} {hits:>6} {rate:>10.4f}")

# ---- Domain breakdown ----
print(f"\n--- Domain Breakdown (threshold=0.25) ---")
domains_sorted = sorted(domain_metrics.keys())
header = f"{'Domain':<15} {'Model':<10} {'Recall@5':>10} {'MRR':>10} {'nDCG@10':>10} {'N':>5}"
print(header)
print("-" * len(header))

for domain in domains_sorted:
    for model_name, label in [("base", "Base E5"), ("ft", "FT E5")]:
        metrics_list = domain_metrics[domain][model_name][0.25]
        if not metrics_list:
            continue
        avg = {
            k: np.mean([m[k] for m in metrics_list])
            for k in ["recall@5", "mrr", "ndcg@10"]
        }
        print(f"{domain:<15} {label:<10} {avg['recall@5']:>10.4f} "
              f"{avg['mrr']:>10.4f} {avg['ndcg@10']:>10.4f} {len(metrics_list):>5}")

print("\nRetrieval evaluation complete.")

## BERTScore Computation

Computes BERTScore F1 for all 6 configs using `bert-base-multilingual-cased`
as the reference model. BERTScore captures semantic similarity beyond lexical
overlap, which is particularly useful for Turkish where morphological variation
causes Token F1 to undercount semantically correct answers.

Results are stored per-question for bootstrap CI computation in the next section.

In [ ]:
from bert_score import score as bert_score_fn

CONFIG_FILES = {
    "C1": "config1_predictions_fixed.json",
    "C2": "config2_finetuned_embeddings_predictions.json",
    "C3": "config3_predictions_fixed.json",
    "C4": "config4_predictions.json",
    "C5": "config5_best_predictions.json",
    "C6": "config6_full_predictions.json",
}

CONFIG_LABELS = {
    "C1": "Baseline",
    "C2": "+ FT Embeddings",
    "C3": "+ FT Embed + Reranker",
    "C4": "+ QLoRA LLM",
    "C5": "+ FT Embed + QLoRA",
    "C6": "Full Pipeline",
}

BERT_MODEL = "bert-base-multilingual-cased"

# Load all prediction files
config_data = {}
for cfg, fname in CONFIG_FILES.items():
    path = RESULTS_DIR / fname
    if not path.exists():
        print(f"[WARNING] {cfg} not found at {path}")
        continue
    with open(path, "r", encoding="utf-8") as f:
        config_data[cfg] = json.load(f)
    print(f"Loaded {cfg}: {len(config_data[cfg]['predictions'])} predictions")

# Compute BERTScore for each config
bertscore_results = {}  # cfg -> {"mean": float, "per_question": list[float]}

for cfg in ["C1", "C2", "C3", "C4", "C5", "C6"]:
    if cfg not in config_data:
        continue
    data = config_data[cfg]
    preds = data["predictions"]
    refs = data["references"]

    print(f"\nComputing BERTScore for {cfg} ({CONFIG_LABELS[cfg]})...")
    P, R, F1 = bert_score_fn(
        preds, refs,
        model_type=BERT_MODEL,
        lang="tr",
        verbose=False,
        batch_size=32,
    )
    f1_list = F1.cpu().numpy().tolist()
    bertscore_results[cfg] = {
        "mean": float(np.mean(f1_list)),
        "per_question": f1_list,
    }
    print(f"  BERTScore F1: {bertscore_results[cfg]['mean']:.4f}")

# Summary table
print("\n" + "=" * 50)
print("BERTScore F1 Summary")
print("=" * 50)
print(f"{'Config':<6} {'Description':<28} {'BERTScore F1':>14}")
print("-" * 50)
for cfg in ["C1", "C2", "C3", "C4", "C5", "C6"]:
    if cfg in bertscore_results:
        print(f"{cfg:<6} {CONFIG_LABELS[cfg]:<28} {bertscore_results[cfg]['mean']:>14.4f}")
    else:
        print(f"{cfg:<6} {CONFIG_LABELS[cfg]:<28} {'MISSING':>14}")

## Bootstrap Confidence Intervals

Computes 95% bootstrap confidence intervals (1000 resamples, percentile method)
for three metrics across all 6 configs:
- Token F1
- ROUGE-L
- BERTScore F1

Per-question scores are computed here (Token F1 and ROUGE-L) rather than
relying on the aggregate means from Notebook 02, so the bootstrap samples
correctly reflect per-question variance.

In [ ]:
from rouge_score import rouge_scorer


def per_question_token_f1(predictions, references):
    """Compute per-question Token F1 scores."""
    scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens = Counter(normalize_turkish(pred).split())
        ref_tokens = Counter(normalize_turkish(ref).split())
        if not pred_tokens or not ref_tokens:
            scores.append(0.0)
            continue
        common = sum((pred_tokens & ref_tokens).values())
        if common == 0:
            scores.append(0.0)
            continue
        precision = common / sum(pred_tokens.values())
        recall = common / sum(ref_tokens.values())
        scores.append(2 * precision * recall / (precision + recall))
    return scores


def per_question_rouge_l(predictions, references):
    """Compute per-question ROUGE-L F1 scores."""
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    return [
        scorer.score(normalize_turkish(ref), normalize_turkish(pred))["rougeL"].fmeasure
        for pred, ref in zip(predictions, references)
    ]


def bootstrap_ci(scores, n_resamples=1000, alpha=0.05, seed=42):
    """Compute percentile bootstrap CI from per-question score list."""
    rng = np.random.default_rng(seed)
    scores_arr = np.array(scores)
    size = len(scores_arr)
    boot_means = []
    for _ in range(n_resamples):
        idx = rng.integers(0, size, size=size)
        boot_means.append(float(np.mean(scores_arr[idx])))
    boot_means.sort()
    lower = boot_means[int(n_resamples * alpha / 2)]
    upper = boot_means[int(n_resamples * (1 - alpha / 2))]
    return {
        "mean": float(np.mean(scores_arr)),
        "lower": lower,
        "upper": upper,
    }


# ---- Compute per-question scores for all configs ----
per_q_scores = {}  # cfg -> {"token_f1": [...], "rouge_l": [...], "bertscore": [...]}

for cfg in ["C1", "C2", "C3", "C4", "C5", "C6"]:
    if cfg not in config_data:
        continue
    data = config_data[cfg]
    preds = data["predictions"]
    refs = data["references"]

    per_q_scores[cfg] = {
        "token_f1": per_question_token_f1(preds, refs),
        "rouge_l": per_question_rouge_l(preds, refs),
        "bertscore": bertscore_results.get(cfg, {}).get("per_question", []),
    }

# ---- Bootstrap CIs ----
print("\n" + "=" * 90)
print("BOOTSTRAP 95% CONFIDENCE INTERVALS (1000 resamples)")
print("=" * 90)

bootstrap_results = {}  # cfg -> metric -> {mean, lower, upper}

header = (f"{'Config':<6} {'Metric':<12} {'Mean':>8} {'95% CI Lower':>14} "
          f"{'95% CI Upper':>14} {'CI Width':>10}")
print(header)
print("-" * len(header))

for cfg in ["C1", "C2", "C3", "C4", "C5", "C6"]:
    if cfg not in per_q_scores:
        continue
    bootstrap_results[cfg] = {}
    for metric_name in ["token_f1", "rouge_l", "bertscore"]:
        scores = per_q_scores[cfg][metric_name]
        if not scores:
            continue
        ci = bootstrap_ci(scores)
        bootstrap_results[cfg][metric_name] = ci
        width = ci["upper"] - ci["lower"]
        print(f"{cfg:<6} {metric_name:<12} {ci['mean']:>8.4f} "
              f"{ci['lower']:>14.4f} {ci['upper']:>14.4f} {width:>10.4f}")

print("\nBootstrap analysis complete.")

## Wilcoxon Signed-Rank Tests

Tests whether pairwise differences between configs are statistically
significant using the Wilcoxon signed-rank test on per-question Token F1
scores. This is a non-parametric paired test appropriate for bounded,
potentially non-normal metrics.

**6 key comparisons:**

| Pair | Tests |
|------|-------|
| C1 vs C2 | Effect of fine-tuned embeddings |
| C1 vs C4 | Effect of QLoRA LLM |
| C2 vs C5 | Sub-additivity check (FT embed alone vs FT embed + QLoRA) |
| C2 vs C3 | Effect of reranker on top of FT embeddings |
| C1 vs C6 | Full pipeline improvement over baseline |
| C5 vs C6 | Effect of reranker on the best non-reranker config |

In [ ]:
PAIRS = [
    ("C1", "C2", "FT Embeddings"),
    ("C1", "C4", "QLoRA LLM"),
    ("C2", "C5", "Sub-additivity (FT+QLoRA vs FT)"),
    ("C2", "C3", "Reranker (on FT embed)"),
    ("C1", "C6", "Full pipeline vs Baseline"),
    ("C5", "C6", "Reranker (on best)"),
]

sig_results = []  # list of result dicts for saving

print("\n" + "=" * 95)
print("WILCOXON SIGNED-RANK TESTS (Token F1, two-sided)")
print("=" * 95)

header = (f"{'Pair':<10} {'Tests':<38} {'Mean A':>8} {'Mean B':>8} "
          f"{'Delta':>8} {'p-value':>10} {'Sig?':>6}")
print(header)
print("-" * len(header))

for cfg_a, cfg_b, description in PAIRS:
    if cfg_a not in per_q_scores or cfg_b not in per_q_scores:
        print(f"{cfg_a}->{cfg_b:<5} {'MISSING DATA':<38}")
        continue

    scores_a = per_q_scores[cfg_a]["token_f1"]
    scores_b = per_q_scores[cfg_b]["token_f1"]

    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)
    delta = mean_b - mean_a

    # Wilcoxon test: check if all differences are zero (degenerate case)
    diffs = [b - a for a, b in zip(scores_a, scores_b)]
    if all(d == 0 for d in diffs):
        stat, p_value = 0.0, 1.0
    else:
        stat, p_value = wilcoxon(scores_a, scores_b)

    sig = "*" if p_value < 0.05 else ""
    if p_value < 0.01:
        sig = "**"
    if p_value < 0.001:
        sig = "***"

    pair_label = f"{cfg_a}->{cfg_b}"
    print(f"{pair_label:<10} {description:<38} {mean_a:>8.4f} {mean_b:>8.4f} "
          f"{delta:>+8.4f} {p_value:>10.6f} {sig:>6}")

    sig_results.append({
        "pair": pair_label,
        "description": description,
        "mean_a": float(mean_a),
        "mean_b": float(mean_b),
        "delta": float(delta),
        "statistic": float(stat),
        "p_value": float(p_value),
        "significant_0.05": p_value < 0.05,
    })

print("\nSignificance: * p<0.05, ** p<0.01, *** p<0.001")
print("Wilcoxon tests complete.")

## Save Results

Persists all computed results to Drive as JSON:
- `retrieval_metrics.json` -- retrieval metrics at all thresholds + madde hits + domain breakdown
- `statistical_analysis.json` -- BERTScore, bootstrap CIs, Wilcoxon tests

In [ ]:
# ---- Retrieval metrics JSON ----
retrieval_out = {
    "thresholds": THRESHOLDS,
    "models": {},
    "madde_hits": {
        "base_e5": madde_hits["base"],
        "ft_e5": madde_hits["ft"],
        "n_questions_with_madde": n_with_madde,
    },
    "domain_breakdown": {},
}

for model_name in ["base", "ft"]:
    model_key = "base_e5" if model_name == "base" else "ft_e5"
    retrieval_out["models"][model_key] = {}
    for thresh in THRESHOLDS:
        metrics_list = retrieval_results[model_name][thresh]
        avg = {
            k: float(np.mean([m[k] for m in metrics_list]))
            for k in ["recall@5", "recall@10", "mrr", "ndcg@10"]
        }
        retrieval_out["models"][model_key][str(thresh)] = avg

for domain in sorted(domain_metrics.keys()):
    retrieval_out["domain_breakdown"][domain] = {}
    for model_name in ["base", "ft"]:
        model_key = "base_e5" if model_name == "base" else "ft_e5"
        metrics_list = domain_metrics[domain][model_name][0.25]
        if metrics_list:
            avg = {
                k: float(np.mean([m[k] for m in metrics_list]))
                for k in ["recall@5", "recall@10", "mrr", "ndcg@10"]
            }
            avg["n"] = len(metrics_list)
            retrieval_out["domain_breakdown"][domain][model_key] = avg

ret_path = RESULTS_DIR / "retrieval_metrics.json"
with open(ret_path, "w", encoding="utf-8") as f:
    json.dump(retrieval_out, f, ensure_ascii=False, indent=2)
print(f"Retrieval metrics saved -> {ret_path}")

# ---- Statistical analysis JSON ----
stat_out = {
    "bertscore": {
        cfg: {"mean": v["mean"]}
        for cfg, v in bertscore_results.items()
    },
    "bootstrap_ci": {
        cfg: metrics
        for cfg, metrics in bootstrap_results.items()
    },
    "wilcoxon_tests": sig_results,
}

stat_path = RESULTS_DIR / "statistical_analysis.json"
with open(stat_path, "w", encoding="utf-8") as f:
    json.dump(stat_out, f, ensure_ascii=False, indent=2)
print(f"Statistical analysis saved -> {stat_path}")

print("\nAll results saved to Drive. Notebook complete.")